# Lab 02 - Notebook 1: Khảo Sát 3 Họ Kiến Trúc Pre-trained & Tiền Xử Lý Dữ Liệu CIFAR-10

- **Purpose**: Khảo sát và so sánh đặc tính cấu trúc của 3 họ mạng nơ-ron tích chập kinh điển (**ResNet-18, VGG-16, DenseNet-121**) đã được huấn luyện trên ImageNet; phân tích số lượng tham số, tầng trích xuất đặc trưng và đầu phân loại; thiết lập quy trình nạp tập dữ liệu CIFAR-10 với chuẩn hóa Z-score ImageNet và Data Augmentation.
- **Roadmap**:

| Step | Module/Task | Description | Output/Artifact |
| :--- | :--- | :--- | :--- |
| **Step 1** | Môi Trường & Khả Năng Tái Lập | Cố định `seed=248`, khởi tạo `torch.device`, cấu hình VRAM $\le 1.8\text{GB}$ | Thiết bị tính toán chuẩn, môi trường ngẫu nhiên xác định |
| **Step 2** | Khảo Sát 3 Kiến Trúc Pre-trained | Nạp ResNet-18, VGG-16, DenseNet-121; kiểm tra tầng cuối và đếm tham số | Bảng đối chuẩn cấu trúc và tham số 3 họ kiến trúc |
| **Step 3** | Tiền Xử Lý Dữ Liệu & ImageNet Stats | Tải CIFAR-10, áp dụng ImageNet Z-score Normalization (mean/std) | `train_dataset`, `test_dataset` |
| **Step 4** | Data Augmentation & DataLoader Engine | Cấu hình `RandomCrop(32, 4)`, `RandomHorizontalFlip()`, `DataLoader` `batch_size=32` | `train_loader`, `test_loader` với `pin_memory=True` |
| **Step 5** | Trực Quan Hóa Dữ Liệu Khảo Sát (EDA) | Kiểm tra tính cân bằng 10 lớp, hiển thị lưới 16 ảnh kèm un-normalization | Biểu đồ phân phối nhãn và lưới 16 ảnh trực quan |

- **References**: [docs/lab2/PIPELINE_EXPLANATION.md](../../docs/lab2/PIPELINE_EXPLANATION.md), [docs/lab2/README.md](../../docs/lab2/README.md), [agents/rules/NOTEBOOK_HEADER_CONVENTION.md](../../.agents/rules/NOTEBOOK_HEADER_CONVENTION.md).

## Step 1: Thiết Lập Môi Trường & Khả Năng Tái Lập (Reproducibility)

Theo quy chuẩn dự án (`PYTORCH_FRAMEWORK_RULES.md`):
- Luôn kiểm tra thiết bị tính toán động: ưu tiên GPU (`cuda`) nếu khả dụng, fallback sang CPU.
- Cố định seed ngẫu nhiên (`seed=248` hoặc `42`) cho toàn bộ `torch`, `torch.cuda`, `numpy`, `random`.
- Kích hoạt `cudnn.deterministic = True` để kết quả huấn luyện hoàn toàn nhất quán.

In [ ]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models

# 1. Cố định seed tái lập thí nghiệm
SEED = 248
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# 2. Thiết lập thiết bị tính toán động
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"[*] Thiết bị tính toán đang sử dụng: {device}")
if device.type == 'cuda':
    print(f"[*] GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"[*] VRAM khả dụng: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")


## Step 2: Khảo Sát 3 Họ Kiến Trúc Pre-trained (ResNet, VGG, DenseNet)

Trong thị giác máy tính, 3 kiến trúc sau đại diện cho 3 triết lý thiết kế mạng CNN kinh điển:
1. **ResNet-18**: Sử dụng Residual Blocks với kết nối tắt (skip connections) $\mathcal{F}(x) + x$, giải quyết triệt để tiêu biến đạo hàm. Tầng cuối là `fc = Linear(512, 1000)`.
2. **VGG-16**: Thiết kế tuần tự tuyến tính truyền thống với tích chập $3\times 3$ nhỏ xếp chồng. Tầng phân loại rất nặng nề với 3 tầng Fully Connected (`Linear(25088, 4096) -> Linear(4096, 4096) -> Linear(4096, 1000)`).
3. **DenseNet-121**: Thiết lập liên kết dày đặc, ghép kênh (concatenation) tất cả các tầng trước vào tầng sau, tối ưu hóa tái sử dụng đặc trưng với tổng tham số rất nhỏ (~8M).

Dưới đây ta nạp cả 3 mô hình từ `torchvision.models` để kiểm tra cấu trúc và đếm tham số:

In [ ]:
def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

print("=" * 70)
print(f"{'Mô Hình':<15} | {'Tổng Tham Số':<15} | {'Tầng Phân Loại Gốc (Head)':<35}")
print("=" * 70)

# Khảo sát ResNet-18
try:
    resnet18 = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
except Exception:
    resnet18 = models.resnet18(pretrained=True)
r18_total, _ = count_parameters(resnet18)
print(f"{'ResNet-18':<15} | {f'{r18_total:,}':<15} | {str(resnet18.fc):<35}")

# Khảo sát VGG-16
try:
    vgg16 = models.vgg16(weights=models.VGG16_Weights.DEFAULT)
except Exception:
    vgg16 = models.vgg16(pretrained=True)
vgg_total, _ = count_parameters(vgg16)
print(f"{'VGG-16':<15} | {f'{vgg_total:,}':<15} | {str(vgg16.classifier[6]):<35}")

# Khảo sát DenseNet-121
try:
    densenet121 = models.densenet121(weights=models.DenseNet121_Weights.DEFAULT)
except Exception:
    densenet121 = models.densenet121(pretrained=True)
dense_total, _ = count_parameters(densenet121)
print(f"{'DenseNet-121':<15} | {f'{dense_total:,}':<15} | {str(densenet121.classifier):<35}")
print("=" * 70)


## Step 3 & 4: Tiền Xử Lý Dữ Liệu CIFAR-10, ImageNet Normalization & Augmentation

> [!IMPORTANT]
> **Quy tắc ImageNet Normalization**: Vì các bộ lọc của mạng pre-trained được tối ưu trên tập ImageNet, ta **bắt buộc** phải chuẩn hóa ảnh đầu vào theo phân phối của ImageNet:
> - `mean = [0.485, 0.456, 0.406]`
> - `std  = [0.229, 0.224, 0.225]`
> Nếu dùng mean/std của chính CIFAR-10, phân phối đầu vào sẽ bị lệch (distribution shift), làm giảm hiệu năng trích xuất đặc trưng của mạng.

**Chiến lược Data Augmentation cho tập Train**:
- `RandomCrop(32, padding=4)`: Tạo tính bất biến dịch chuyển (translation invariance).
- `RandomHorizontalFlip(p=0.5)`: Tạo tính bất biến phản chiếu đối xứng.
- Tập Test/Validation **không áp dụng** phép tăng cường hình học mà chỉ chuẩn hóa.

In [ ]:
# Định nghĩa transforms
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

# Nạp dữ liệu CIFAR-10
data_dir = './data'
os.makedirs(data_dir, exist_ok=True)

train_dataset = torchvision.datasets.CIFAR10(
    root=data_dir, train=True, download=True, transform=train_transform
)
test_dataset = torchvision.datasets.CIFAR10(
    root=data_dir, train=False, download=True, transform=test_transform
)

# Khởi tạo DataLoader tối ưu GPU
BATCH_SIZE = 32
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True if device.type == 'cuda' else False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True if device.type == 'cuda' else False
)

classes = ('airplane', 'automobile', 'bird', 'cat', 'deer',
           'dog', 'frog', 'horse', 'ship', 'truck')

print(f"[*] Số lượng mẫu Train: {len(train_dataset):,}")
print(f"[*] Số lượng mẫu Test:  {len(test_dataset):,}")
print(f"[*] Kích thước mỗi batch: {BATCH_SIZE} ảnh")
print(f"[*] 10 lớp phân loại: {classes}")


## Step 5: Trực Quan Hóa Dữ Liệu Khảo Sát (EDA) & Lưới 16 Ảnh Mẫu

Để kiểm tra ảnh sau khi đã được tiền xử lý, ta định nghĩa hàm đảo ngược chuẩn hóa (un-normalization):
$$x_{\text{orig}} = x_{\text{norm}} \cdot \sigma + \mu$$
và vẽ lưới 16 ảnh kèm nhãn thực tế.

In [ ]:
def imshow_unnorm(tensor, title=None):
    # tensor: [C, H, W]
    inp = tensor.clone().detach().cpu().numpy().transpose((1, 2, 0))
    mean = np.array(IMAGENET_MEAN)
    std = np.array(IMAGENET_STD)
    inp = std * inp + mean
    inp = np.clip(inp, 0, 1)
    return inp

# Lấy 1 batch từ train_loader
dataiter = iter(train_loader)
images, labels = next(dataiter)

plt.figure(figsize=(10, 10))
for idx in range(16):
    ax = plt.subplot(4, 4, idx + 1)
    img_disp = imshow_unnorm(images[idx])
    plt.imshow(img_disp)
    plt.title(f"{classes[labels[idx]]}", fontsize=10, color='darkblue')
    plt.axis('off')

plt.suptitle("Lưới 16 Ảnh Mẫu CIFAR-10 Sau Data Augmentation & Normalization", fontsize=13, weight='bold')
plt.tight_layout()
plt.show()

# Đếm phân phối 10 lớp
from collections import Counter
train_counts = Counter(train_dataset.targets)
print("[*] Phân phối số lượng mẫu trên 10 lớp:")
for cls_idx, count in sorted(train_counts.items()):
    print(f"    - Lớp {cls_idx} ({classes[cls_idx]:<11}): {count:,} ảnh")
